In [9]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from scipy.ndimage import gaussian_filter

def plot_kmeans_with_scree(n_samples=250, n_clusters=4, max_k=8, random_state=42, 
                            n_points=600, sigma=2.0, save_path="beamer/images/clustering_with_scree.png"):
    """
    Fits a K-Means model on asymmetric clusters with distinct variances and generates smooth 
    decision boundaries alongside a Scree plot with positioned legends inside each plot.
    """
    rng = np.random.default_rng(random_state)

    # 1. Non-symmetrical cluster centers
    cluster_centers = np.array([
        [-3.0, -2.0],
        [ 1.0,  3.0],
        [ 3.5, -1.0],
        [-1.0,  1.5]
    ])
    
    # 2. Distinct standard deviation (variance) for EACH cluster
    cluster_stds = [0.5, 1.0, 1.6, 2.2]
    
    # 3. Generate cluster data
    samples_per_cluster = [60, 70, 60, 60]
    X_list = []
    for center, std, n in zip(cluster_centers, cluster_stds, samples_per_cluster):
        cluster_data = rng.normal(loc=center, scale=std, size=(n, 2))
        X_list.append(cluster_data)
    
    X = np.vstack(X_list)

    # Calculate Inertia across K=1..max_k for Scree Plot
    k_values = list(range(1, max_k + 1))
    inertias = []
    for k in k_values:
        km_test = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        km_test.fit(X)
        inertias.append(km_test.inertia_)

    # Fit target KMeans model (k=4)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(X)

    # High-resolution grid for ultra-smooth contours
    x1_range = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, n_points)
    x2_range = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, n_points)
    x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
    grid_points = np.column_stack((x1_grid.ravel(), x2_grid.ravel()))

    # Compute distances to all centroids to create smooth boundaries
    distances = kmeans.transform(grid_points)
    min_dist_diff = np.min(distances, axis=1) - np.partition(distances, 1, axis=1)[:, 1]
    dist_grid = min_dist_diff.reshape(x1_grid.shape)

    # Apply Gaussian smoothing to eliminate grid aliasing and sharp artifacts
    smooth_grid = gaussian_filter(dist_grid, sigma=sigma)

    # Create 1x2 Subplots figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Clustering', 'Elbow plot'),
        horizontal_spacing=0.12
    )

    # --- SUBPLOT 1: CLUSTERING ---
    fig.add_trace(go.Contour(
        x=x1_range, y=x2_range, z=smooth_grid,
        showscale=False,
        contours=dict(
            coloring='none',
            showlines=True,
            start=0,
            end=0,
            size=0
        ),
        line=dict(width=2.5, color='black', dash='dash'),
        showlegend=False
    ), row=1, col=1)

    colors = ['red', 'green', 'blue', 'orange']
    symbols = ['circle', 'square', 'triangle-up', 'diamond']
    
    for cls in range(n_clusters):
        mask = kmeans.labels_ == cls
        fig.add_trace(go.Scatter(
            x=X[mask, 0], y=X[mask, 1],
            mode='markers',
            opacity=0.4,
            marker=dict(
                size=8,
                symbol=symbols[cls % len(symbols)],
                color=colors[cls % len(colors)],
                line=dict(width=1, color='black')
            ),
            name=f'Cluster {cls + 1}',
            legend='legend'  # Използва първата легенда
        ), row=1, col=1)

    centers = kmeans.cluster_centers_
    fig.add_trace(go.Scatter(
        x=centers[:, 0], y=centers[:, 1],
        mode='markers',
        marker=dict(
            size=10,
            symbol='cross',
            color='black',
            line=dict(width=0.1, color='black')
        ),
        name='Centroids',
        legend='legend'  # Използва първата легенда
    ), row=1, col=1)

    # --- SUBPLOT 2: SCREE PLOT ---
    fig.add_trace(go.Scatter(
        x=k_values,
        y=inertias,
        mode='lines+markers',
        marker=dict(size=10, color='royalblue'),
        line=dict(width=2.5, color='royalblue'),
        name='WCSS',
        legend='legend2'  # Използва втората легенда
    ), row=1, col=2)

    optimal_inertia = inertias[n_clusters - 1]
    fig.add_trace(go.Scatter(
        x=[n_clusters],
        y=[optimal_inertia],
        mode='markers',
        marker=dict(size=14, color='red', symbol='circle'),
        name=f'Optimal K={n_clusters}',
        legend='legend2'  # Използва втората легенда
    ), row=1, col=2)

    # Настройки на оформлението и позициите на двете легенди
    fig.update_layout(
        width=1200,
        height=400,
        plot_bgcolor='white',
        margin=dict(t=60, b=40, l=40, r=40),
        
        # Легенда за първата графика (Clustering) - позиционирана вътре долу-вдясно или горе-вляво
        legend=dict(
            font=dict(size=12),
            x=0.02, y=0.98,
            xanchor='left', yanchor='top',
            bgcolor='rgba(255,255,255,0.7)',
            bordercolor='lightgray',
            borderwidth=1
        ),
        
        # Легенда за втората графика (Elbow plot) - позиционирана вътре горе-вдясно
        legend2=dict(
            font=dict(size=12),
            x=0.98, y=0.98,
            xanchor='right', yanchor='top',
            bgcolor='rgba(255,255,255,0.7)',
            bordercolor='lightgray',
            borderwidth=1
        )
    )

    fig.for_each_annotation(lambda a: a.update(font=dict(size=20)))

    fig.update_xaxes(title_text="X1", title_font=dict(size=16), row=1, col=1)
    fig.update_yaxes(title_text="X2", title_font=dict(size=16), row=1, col=1)
    fig.update_xaxes(title_text="Number of Clusters (K)", tickvals=k_values, title_font=dict(size=16), row=1, col=2)
    fig.update_yaxes(title_text="WCSS", title_font=dict(size=16), row=1, col=2)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.write_image(save_path)
    fig.show()

    return fig, kmeans

In [10]:
plot_kmeans_with_scree(random_state=4)

(Figure({
     'data': [{'contours': {'coloring': 'none', 'end': 0, 'showlines': True, 'size': 0, 'start': 0},
               'line': {'color': 'black', 'dash': 'dash', 'width': 2.5},
               'showlegend': False,
               'showscale': False,
               'type': 'contour',
               'x': {'bdata': ('nKeHMaj6HcBAHI4ze94dwOOQlDVOwh' ... 'u1CiLdIUD48LGJOOshQKa2rghP+SFA'),
                     'dtype': 'f8'},
               'xaxis': 'x',
               'y': {'bdata': ('RD3pyEYGFcAuWaAUX/EUwBh1V2B33B' ... 'kSpRC6G0CijVtZ+M4bQLdxpA3g4xtA'),
                     'dtype': 'f8'},
               'yaxis': 'y',
               'z': {'bdata': ('wGr34RsPCsAWf/Ci5BMKwMZ2vwl3HA' ... 'L1nUI48b9b2xenER3xv6jCSpbnDfG/'),
                     'dtype': 'f8',
                     'shape': '600, 600'}},
              {'legend': 'legend',
               'marker': {'color': 'red', 'line': {'color': 'black', 'width': 1}, 'size': 8, 'symbol': 'circle'},
               'mode': 'markers',
       

In [42]:
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

def plot_pca_pipeline(output_path="./beamer/images/pca.png"):

    # Generate 3D data with variance/noise
    rng = np.random.default_rng(7)
    latent = rng.normal(size=(250, 2))
    basis = np.array([[1.0, 0.2], [0.4, 1.0], [0.8, -0.6]])
    X_3d = latent @ basis.T + rng.normal(scale=0.45, size=(250, 3))

    # Fit PCA for all components and extract 2D projection
    pca_full = PCA().fit(X_3d)
    pca_2d = PCA(n_components=2)
    X_pca = pca_2d.fit_transform(X_3d)
    
    var_exp = pca_full.explained_variance_ratio_
    cum_var_exp = np.cumsum(var_exp)

    # Subplot configuration
    fig = make_subplots(
        rows=2,
        cols=2,
        row_heights=[0.75, 0.25],
        column_widths=[0.55, 0.45],
        specs=[
            [{"type": "scene"}, {"type": "xy"}],
            [{"type": "xy", "colspan": 2}, None]
        ],
        subplot_titles=(
            "3D Data",
            "2D Projection",
            "Scree Plot"
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.10
    )

    # 1. Original 3D Scatter Plot
    fig.add_trace(
        go.Scatter3d(
            x=X_3d[:, 0],
            y=X_3d[:, 1],
            z=X_3d[:, 2],
            mode="markers",
            marker=dict(size=3, color="#636EFA", opacity=0.18),
            name="Data Points"
        ),
        row=1, col=1
    )

    # Add 3D Eigenvector arrows
    mean = pca_full.mean_
    components = pca_full.components_
    eigenvalues = pca_full.explained_variance_
    vector_colors = ["#EF553B", "#00CC96", "#AB63FA"]

    for i in range(3):
        vector = components[i] * np.sqrt(eigenvalues[i]) * 3.2
        tip = mean + vector

        # Line shaft
        fig.add_trace(
            go.Scatter3d(
                x=[mean[0], tip[0]],
                y=[mean[1], tip[1]],
                z=[mean[2], tip[2]],
                mode="lines",
                line=dict(color=vector_colors[i], width=4.0),
                name=f"Eigenvector {i+1}"
            ),
            row=1, col=1
        )

        # Arrowhead cone
        fig.add_trace(
            go.Cone(
                x=[tip[0]],
                y=[tip[1]],
                z=[tip[2]],
                u=[vector[0]],
                v=[vector[1]],
                w=[vector[2]],
                sizemode="absolute",
                sizeref=0.25,
                colorscale=[[0, vector_colors[i]], [1, vector_colors[i]]],
                showscale=False,
                anchor="tip"
            ),
            row=1, col=1
        )

    # 2. PCA 2D Projected Scatter Plot
    fig.add_trace(
        go.Scatter(
            x=X_pca[:, 0],
            y=X_pca[:, 1],
            mode="markers",
            marker=dict(size=3.5, color="#636EFA", opacity=0.35),
            name="PCA Scores"
        ),
        row=1, col=2
    )

    # 3. Scree Plot
    components_idx = ["PC1", "PC2", "PC3"]
    
    fig.add_trace(
        go.Bar(
            x=components_idx,
            y=var_exp,
            name="Individual Variance",
            text=[f"{v:.1%}" for v in var_exp],
            textposition="auto",
            marker_color="#636EFA",
            opacity=0.75
        ),
        row=2, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=components_idx,
            y=cum_var_exp,
            mode="lines+markers+text",
            name="Cumulative Variance",
            text=[f"{v:.1%}" for v in cum_var_exp],
            textposition="top center",
            line=dict(color="#EF553B", width=2)
        ),
        row=2, col=1
    )

    # Layout configuration
    fig.update_layout(
        width=1000,
        height=520,
        template="plotly_white",
        showlegend=False,
        margin=dict(l=30, r=30, t=40, b=30),
        title={
            'text': "PCA",
            'x': 0.5,
            'xanchor': 'center',
        }
    )

    # Фиксиране на заглавието "3D Data" по-нагоре, за да не се застъпва
    fig.layout.annotations[0].update(y=0.98)

    # 3D Scene setup - Запълване на цялото вертикално пространство (от 0.23 до 0.98)
    fig.update_scenes(
        xaxis_title="X1",
        yaxis_title="X2",
        zaxis_title="X3",
        aspectmode="cube",
        camera=dict(
            eye=dict(x=1.5, y=-1.5, z=0.55),  # Приближаваме още малко камерата за увеличение
            center=dict(x=0, y=0, z=0)
        ),
        row=1, col=1
    )

    # Zoomed out 2D projection axes
    fig.update_xaxes(title_text="PC1", range=[-8, 8], row=1, col=2)
    fig.update_yaxes(title_text="PC2", range=[-8, 8], row=1, col=2)

    # Scree Plot Axes
    fig.update_xaxes(row=2, col=1)
    fig.update_yaxes(range=[0, 1.45], row=2, col=1)

    # Save output file
    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.write_image(output_path)

    fig.show()

# Run script
plot_pca_pipeline()